In [46]:
%pip install -qU "google-cloud-aiplatform[agent_engines,adk]>=1.112"

In [47]:
from google.colab import auth
auth.authenticate_user()

import os
import vertexai

PROJECT_ID = "qwiklabs-gcp-03-918ac2e61dcf"
LOCATION = "us-central1"
CONTACT_EMAIL = "david.bullard364@gmail.com"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["WEATHER_CONTACT"] = CONTACT_EMAIL

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")

Project: qwiklabs-gcp-03-918ac2e61dcf
Location: us-central1


In [48]:
!gcloud config set project {PROJECT_ID}
!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [49]:
import json
import time

from functools import lru_cache
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen


USER_AGENT = f"ADKWeatherChallenge/1.0 ({CONTACT_EMAIL})"
REQUEST_TIMEOUT = 20

_last_geocode_request = 0.0

In [50]:
def _fetch_json(
    url: str,
    accept: str = "application/geo+json",
) -> dict:
    """Retrieve and decode a JSON document."""
    request = Request(
        url,
        headers={
            "User-Agent": USER_AGENT,
            "Accept": accept,
        },
    )

    with urlopen(request, timeout=REQUEST_TIMEOUT) as response:
        return json.load(response)


@lru_cache(maxsize=256)
def _get_coordinates(
    city: str,
    state: str,
) -> tuple[float, float]:
    """Convert a U.S. city and state into latitude and longitude."""
    global _last_geocode_request

    elapsed = time.monotonic() - _last_geocode_request
    time.sleep(max(0.0, 1.0 - elapsed))

    parameters = urlencode(
        {
            "city": city,
            "state": state,
            "countrycodes": "us",
            "format": "jsonv2",
            "limit": 1,
        }
    )

    url = (
        "https://nominatim.openstreetmap.org/search?"
        f"{parameters}"
    )

    try:
        locations = _fetch_json(
            url,
            accept="application/json",
        )
    finally:
        _last_geocode_request = time.monotonic()

    if not locations:
        raise ValueError(
            f"No U.S. location found for {city}, {state}."
        )

    return (
        float(locations[0]["lat"]),
        float(locations[0]["lon"]),
    )

In [51]:
def _measurement_value(properties: dict, field: str):
    """Extract a measurement value from an NWS observation."""
    measurement = properties.get(field) or {}
    return measurement.get("value")


def _condition_advisory(
    temperature_f: float | None,
    wind_mph: float | None,
    official_alerts: list[dict],
) -> str:
    """Create an alert or advisory from current conditions."""
    if official_alerts:
        alert_names = [
            alert.get("event")
            for alert in official_alerts
            if alert.get("event")
        ]

        if alert_names:
            return "Official NWS alert: " + "; ".join(alert_names)

    advisories = []

    if temperature_f is not None:
        if temperature_f >= 95:
            advisories.append("high heat conditions")
        elif temperature_f <= 32:
            advisories.append("freezing conditions")

    if wind_mph is not None and wind_mph >= 30:
        advisories.append("strong winds")

    if advisories:
        return "Use caution due to " + "; ".join(advisories) + "."

    return (
        "No active NWS alert or threshold-based "
        "weather advisory."
    )

In [52]:
def get_current_weather(city: str, state: str) -> dict:
    """
    Get current weather, the next forecast period, and active weather
    alerts for a city in the United States.

    Args:
        city: U.S. city name, such as Austin or Seattle.
        state: State name or two-letter abbreviation, such as Texas
            or TX.

    Returns:
        A dictionary containing the current observation, forecast,
        official NWS alerts, and a weather advisory.
    """
    city = city.strip()
    state = state.strip()

    if not city or not state:
        return {
            "status": "error",
            "message": "Both city and state are required.",
        }

    try:
        latitude, longitude = _get_coordinates(city, state)

        # Locate the appropriate NWS endpoints.
        point_url = (
            "https://api.weather.gov/points/"
            f"{latitude:.4f},{longitude:.4f}"
        )
        point = _fetch_json(point_url)
        point_properties = point["properties"]

        # Obtain nearby weather stations.
        stations = _fetch_json(
            point_properties["observationStations"]
        )
        station_features = stations.get("features", [])

        if not station_features:
            raise ValueError(
                "No NWS observation station was found."
            )

        # Try multiple stations because an individual station may
        # have missing or temporarily unavailable observations.
        observation_properties = None
        selected_station = None

        for station in station_features[:5]:
            station_url = station.get("id")

            if not station_url:
                continue

            try:
                observation = _fetch_json(
                    f"{station_url}/observations/latest"
                )
                candidate = observation["properties"]

                candidate_temperature = _measurement_value(
                    candidate,
                    "temperature",
                )

                if candidate_temperature is not None:
                    observation_properties = candidate
                    selected_station = station_url.rsplit("/", 1)[-1]
                    break

            except (
                HTTPError,
                URLError,
                TimeoutError,
                KeyError,
                TypeError,
            ):
                continue

        if observation_properties is None:
            raise ValueError(
                "Nearby NWS stations did not report a usable "
                "current temperature."
            )

        temperature_c = _measurement_value(
            observation_properties,
            "temperature",
        )
        wind_mps = _measurement_value(
            observation_properties,
            "windSpeed",
        )
        humidity = _measurement_value(
            observation_properties,
            "relativeHumidity",
        )

        temperature_f = round(
            (temperature_c * 9 / 5) + 32,
            1,
        )

        wind_mph = (
            round(wind_mps * 2.23694, 1)
            if wind_mps is not None
            else None
        )

        # Retrieve the next forecast period.
        forecast = _fetch_json(point_properties["forecast"])
        forecast_periods = forecast["properties"]["periods"]

        if not forecast_periods:
            raise ValueError("No NWS forecast periods were returned.")

        next_period = forecast_periods[0]

        # Retrieve active official alerts for the location.
        alerts_url = (
            "https://api.weather.gov/alerts/active?"
            + urlencode(
                {
                    "point": (
                        f"{latitude:.4f},{longitude:.4f}"
                    )
                }
            )
        )
        alerts_document = _fetch_json(alerts_url)

        official_alerts = []

        for feature in alerts_document.get("features", [])[:5]:
            properties = feature.get("properties", {})

            official_alerts.append(
                {
                    "event": properties.get("event"),
                    "severity": properties.get("severity"),
                    "headline": properties.get("headline"),
                    "expires": properties.get("expires"),
                }
            )

        advisory = _condition_advisory(
            temperature_f=temperature_f,
            wind_mph=wind_mph,
            official_alerts=official_alerts,
        )

        return {
            "status": "success",
            "location": f"{city}, {state}",
            "coordinates": {
                "latitude": latitude,
                "longitude": longitude,
            },
            "current": {
                "station": selected_station,
                "observed_at": observation_properties.get(
                    "timestamp"
                ),
                "description": (
                    observation_properties.get("textDescription")
                    or "No text description reported"
                ),
                "temperature_f": temperature_f,
                "relative_humidity_percent": (
                    round(humidity, 1)
                    if humidity is not None
                    else None
                ),
                "wind_mph": wind_mph,
                "wind_direction_degrees": _measurement_value(
                    observation_properties,
                    "windDirection",
                ),
            },
            "next_forecast": {
                "name": next_period.get("name"),
                "temperature": next_period.get("temperature"),
                "temperature_unit": next_period.get(
                    "temperatureUnit"
                ),
                "wind": (
                    f"{next_period.get('windSpeed')} "
                    f"{next_period.get('windDirection')}"
                ),
                "summary": next_period.get("shortForecast"),
                "details": next_period.get("detailedForecast"),
            },
            "official_alerts": official_alerts,
            "advisory": advisory,
            "sources": [
                "National Weather Service",
                "OpenStreetMap Nominatim",
            ],
        }

    except HTTPError as error:
        return {
            "status": "error",
            "location": f"{city}, {state}",
            "message": (
                f"Weather service returned HTTP {error.code}: "
                f"{error.reason}"
            ),
        }

    except (URLError, TimeoutError) as error:
        return {
            "status": "error",
            "location": f"{city}, {state}",
            "message": (
                f"Could not reach a weather service: {error}"
            ),
        }

    except (
        KeyError,
        IndexError,
        TypeError,
        ValueError,
    ) as error:
        return {
            "status": "error",
            "location": f"{city}, {state}",
            "message": str(error),
        }

In [53]:
tool_result = get_current_weather("Austin", "Texas")

assert tool_result["status"] == "success", tool_result

print(json.dumps(tool_result, indent=2))

{
  "status": "success",
  "location": "Austin, Texas",
  "coordinates": {
    "latitude": 30.2649261,
    "longitude": -97.7474086
  },
  "current": {
    "station": "KAUS",
    "observed_at": "2026-10-26T14:15:00+00:00",
    "description": "Clear",
    "temperature_f": 64.4,
    "relative_humidity_percent": 63.6,
    "wind_mph": 16.6,
    "wind_direction_degrees": 30
  },
  "next_forecast": {
    "name": "Today",
    "temperature": 84,
    "temperature_unit": "F",
    "wind": "0 to 5 mph ESE",
    "summary": "Mostly Sunny",
    "details": "Mostly sunny, with a high near 84. East southeast wind 0 to 5 mph."
  },
  "official_alerts": [],
  "advisory": "No active NWS alert or threshold-based weather advisory.",
  "sources": [
    "National Weather Service",
    "OpenStreetMap Nominatim"
  ]
}


In [54]:
from google import genai


CANDIDATE_MODELS = [
    "gemini-3.1-flash-lite",
    "gemini-3.5-flash-lite",
    "gemini-3.5-flash",
    # Legacy Qwiklabs possibilities:
    "gemini-2.5-flash",
    "gemini-2.0-flash",
]

CANDIDATE_LOCATIONS = [
    "us-central1",
    "global",
]


def find_available_model():
    """Find a model and location available to this project."""
    failures = []

    for location in CANDIDATE_LOCATIONS:
        client = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location=location,
        )

        for model_id in CANDIDATE_MODELS:
            try:
                response = client.models.generate_content(
                    model=model_id,
                    contents="Respond with only the word OK.",
                )

                print(
                    f"AVAILABLE: {model_id} in {location}"
                )
                print("Test response:", response.text)

                return model_id, location

            except Exception as error:
                error_summary = str(error).split("\n")[0]

                print(
                    f"Unavailable: {model_id} in "
                    f"{location}"
                )

                failures.append(
                    {
                        "model": model_id,
                        "location": location,
                        "error": error_summary,
                    }
                )

    raise RuntimeError(
        "None of the candidate Gemini models were available. "
        "Check the Qwiklabs instructions for the required model."
    )


MODEL_ID, LOCATION = find_available_model()

print("\nSelected configuration")
print("Model:", MODEL_ID)
print("Location:", LOCATION)

Unavailable: gemini-3.1-flash-lite in us-central1
Unavailable: gemini-3.5-flash-lite in us-central1
Unavailable: gemini-3.5-flash in us-central1
AVAILABLE: gemini-2.5-flash in us-central1
Test response: OK

Selected configuration
Model: gemini-2.5-flash
Location: us-central1


In [55]:
from google.adk.agents import Agent


weather_agent = Agent(
    name="weather_agent",
    model=MODEL_ID,
    description=(
        "Provides current U.S. weather observations, forecasts, "
        "and weather alerts."
    ),
    instruction="""
You are a U.S. weather assistant.

When the user asks about weather:

1. Require both a city and state.
2. Always call get_current_weather.
3. Never invent current conditions, forecasts, or alerts.
4. Report the current temperature, conditions, and wind.
5. Report the next forecast period.
6. Report any official alert or condition-based advisory.
7. If the tool returns status="error", clearly explain the error.
""",
    tools=[get_current_weather],
)

print("Agent created with model:", MODEL_ID)

Agent created with model: gemini-2.5-flash


In [56]:
import uuid

from IPython.display import Markdown, display
from vertexai.agent_engines import AdkApp


# Create the application only once.
app = AdkApp(agent=weather_agent)


def _as_dictionary(value) -> dict:
    """Normalize an ADK dictionary or Pydantic object."""
    if isinstance(value, dict):
        return value

    if hasattr(value, "model_dump"):
        return value.model_dump(
            mode="json",
            exclude_none=True,
        )

    raise TypeError(
        f"Unsupported ADK event type: {type(value).__name__}"
    )


def _parts_from_event(event) -> list[dict]:
    """Return normalized content parts from an ADK event."""
    event_dictionary = _as_dictionary(event)
    content = event_dictionary.get("content") or {}

    if not isinstance(content, dict):
        content = _as_dictionary(content)

    parts = content.get("parts") or []

    return [
        part if isinstance(part, dict) else _as_dictionary(part)
        for part in parts
    ]


async def ask_weather_agent(
    city: str,
    state: str,
) -> dict:
    """Ask the weather agent about one U.S. city."""
    user_id = "weather-challenge-test"
    session_id = str(uuid.uuid4())

    await app.async_create_session(
        user_id=user_id,
        session_id=session_id,
    )

    events = []
    text_fragments = []
    tool_calls = []

    prompt = (
        f"What is the current weather in {city}, {state}? "
        "Include the next forecast period and any weather alert "
        "or advisory."
    )

    async for event in app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message=prompt,
    ):
        normalized_event = _as_dictionary(event)
        events.append(normalized_event)

        for part in _parts_from_event(event):
            text = part.get("text")

            if text:
                text_fragments.append(text)

            function_call = (
                part.get("function_call")
                or part.get("functionCall")
            )

            if function_call:
                tool_name = function_call.get("name")

                if tool_name:
                    tool_calls.append(tool_name)

    return {
        "city": city,
        "state": state,
        "answer": "\n".join(text_fragments).strip(),
        "tool_calls": tool_calls,
        "events": events,
    }

In [57]:
result = await ask_weather_agent(
    "Austin",
    "Texas",
)

display(
    Markdown(
        result["answer"]
        or "*The agent returned no textual response.*"
    )
)

print("Tools called:", result["tool_calls"])

assert result["answer"], (
    "The agent returned no answer."
)

assert "get_current_weather" in result["tool_calls"], (
    "The agent did not call get_current_weather."
)

The current weather in Austin, Texas is Clear, with a temperature of 64.4°F and wind blowing at 16.6 mph in the direction of 30 degrees.

The forecast for today is mostly sunny, with a high near 84°F and an east southeast wind of 0 to 5 mph.

There are no active weather alerts or advisories at this time.

Tools called: ['get_current_weather']


In [58]:
TEST_CITIES = [
    ("Austin", "Texas"),
    ("Seattle", "Washington"),
    ("Miami", "Florida"),
]


def test_live_weather_tool_multiple_cities():
    """Live integration test using OpenStreetMap and NWS."""
    results = []

    for city, state in TEST_CITIES:
        result = get_current_weather(city, state)

        assert result["status"] == "success", result
        assert result["location"] == f"{city}, {state}"
        assert result["current"]["observed_at"]
        assert result["current"]["temperature_f"] is not None
        assert result["current"]["description"]
        assert result["next_forecast"]["name"]
        assert (
            result["next_forecast"]["summary"]
            or result["next_forecast"]["details"]
        )
        assert result["advisory"]
        assert "official_alerts" in result

        results.append(result)

        print(
            f"PASS: {city}, {state} | "
            f"{result['current']['temperature_f']}°F | "
            f"{result['current']['description']} | "
            f"{result['advisory']}"
        )

    return results


tool_test_results = test_live_weather_tool_multiple_cities()

PASS: Austin, Texas | 64.4°F | Clear | No active NWS alert or threshold-based weather advisory.
PASS: Seattle, Washington | 50.0°F | Light Rain | No active NWS alert or threshold-based weather advisory.
PASS: Miami, Florida | 80.6°F | Partly Cloudy | Official NWS alert: Coastal Flood Warning; Rip Current Statement; Wind Advisory


In [59]:
async def test_weather_agent_multiple_cities():
    """Verify that the ADK agent calls its tool for each city."""
    results = []

    for city, state in TEST_CITIES:
        result = await ask_weather_agent(city, state)
        answer_lower = result["answer"].lower()

        assert result["answer"], (
            f"The agent returned no text for {city}, {state}."
        )

        assert "get_current_weather" in result["tool_calls"], (
            f"The agent did not call get_current_weather for "
            f"{city}, {state}. Calls: {result['tool_calls']}"
        )

        assert city.lower() in answer_lower, (
            f"The response did not identify {city}."
        )

        temperature_present = any(
            term in answer_lower
            for term in (
                "temperature",
                "degrees",
                "°f",
                "° f",
            )
        )
        forecast_present = any(
            term in answer_lower
            for term in (
                "forecast",
                "today",
                "tonight",
                "later",
            )
        )
        alert_present = any(
            term in answer_lower
            for term in (
                "alert",
                "advisory",
                "warning",
                "watch",
            )
        )

        assert temperature_present, (
            f"No temperature found in the response for {city}."
        )
        assert forecast_present, (
            f"No forecast found in the response for {city}."
        )
        assert alert_present, (
            f"No alert or advisory status found for {city}."
        )

        results.append(result)

        print(f"\nPASS: Agent test for {city}, {state}")
        print("-" * 60)
        print(result["answer"])

    return results

In [60]:
agent_test_results = (
    await test_weather_agent_multiple_cities()
)

print("\n" + "=" * 60)
print(
    f"All {len(agent_test_results)} multi-city "
    "agent tests passed."
)
print("=" * 60)


PASS: Agent test for Austin, Texas
------------------------------------------------------------
The current weather in Austin, Texas is clear with a temperature of 64.4°F and wind blowing at 16.6 mph in the direction of 30 degrees. The relative humidity is 63.6%.

The next forecast period, "Today," predicts mostly sunny skies with a high near 84°F. The wind will be from the east-southeast at 0 to 5 mph.

There are no active National Weather Service alerts or threshold-based weather advisories.

PASS: Agent test for Seattle, Washington
------------------------------------------------------------
The current weather in Seattle, Washington is 50°F with light rain and no wind.

For today, the forecast is a chance of light rain and mostly cloudy skies, with a high near 55°F, falling to around 53°F in the afternoon. A north wind of around 5 mph is expected, with a 40% chance of precipitation and new rainfall amounts of less than a tenth of an inch possible.

There are no active National Wea